# C1 — decomposing the defence, and an adversary that beats trust

The 100-target sweep gave H1 and H2 cleanly: plain experience retrieval goes from 0.270
to 0.560 attack success as the poison rate rises, the defended store stays flat near
0.27, and macro-F1 holds at 0.33 instead of collapsing to 0.17. Two things in it are not
yet reportable.

**The defences were only tested together.** `gated_memory` turns on C2 and C3 at once, so
the table cannot say which one did the work. `gate_only` and `trust_only` run each alone.

**The adversary was easy on C3.** The fakes cite known misinformation domains, which the
system has never seen -- and a Beta(1, 3) prior puts an unseen domain at 0.25, below the
0.3 floor. C3 rejected every poisoned record *for being a stranger*, without ever judging
what it said. `poison kept 0.00` is that, not a content check. An adversary who reads the
system's history and cites its best-established sources defeats C3 by construction and
leaves C2 alone. That is the number that belongs in the report.

The cache from the previous sweep is restored below, so the nine cells already run replay
instantly and only the new ones cost GPU time.

In [ ]:
import os
import pathlib
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/TERABYTE05/VeriMem.git", str(REPO)],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

RESULTS = "/kaggle/working/results"
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"
os.environ["VERIMEM_RESULTS_DIR"] = RESULTS
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## Inputs, and the warm cache

Restoring `cache/llm` and `cache/evaluator` from the previous sweep is what makes this
affordable: every cell that ran before replays from disk, so the session spends its hours
on the cells that are actually new (rule 1).

In [ ]:
import pathlib
import shutil

DATA = pathlib.Path("/kaggle/working/data/averitec")
TRAJ = pathlib.Path("/kaggle/working/data/trajectories")
for d in (DATA, TRAJ):
    d.mkdir(parents=True, exist_ok=True)

print("attached:", [p.name for p in pathlib.Path("/kaggle/input").iterdir()])

for name, dest in (
    ("dev.json", DATA), ("pool_dev.jsonl", DATA), ("misinfo_list.txt", DATA),
    ("trajectories.jsonl", TRAJ),
):
    if (dest / name).exists():
        continue
    for found in pathlib.Path("/kaggle/input").rglob(name):
        shutil.copy(found, dest / name)
        break
    else:
        raise SystemExit(f"{name} not found among the attached inputs")

for p in sorted(DATA.iterdir()) + sorted(TRAJ.iterdir()):
    print(f"  {p.name:<24} {p.stat().st_size/1e6:7.1f} MB")

restored = 0
for ns in ("llm", "evaluator"):
    dst = pathlib.Path(f"/kaggle/working/cache/{ns}")
    if dst.exists():
        continue
    for src in pathlib.Path("/kaggle/input").rglob(f"cache/{ns}"):
        if src.is_dir():
            shutil.copytree(src, dst)
            n = len(list(dst.rglob("*.json")))
            restored += n
            print(f"restored {n} cached {ns} calls")
            break
if not restored:
    print("WARNING: cold cache. Every call will be recomputed; expect the full runtime.")

In [ ]:
!pip install -q transformers datasets accelerate openai pyyaml python-dotenv 2>&1 | tail -2

## The evaluator, from the attached G1 output

In [ ]:
def find_evaluator():
    for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
        if not root.exists():
            continue
        for p in root.rglob("model/config.json"):
            if (p.parent / "model.safetensors").exists() or (p.parent / "pytorch_model.bin").exists():
                return p.parent
    return None


EVALUATOR = find_evaluator()
if EVALUATOR is None:
    raise SystemExit(
        "no evaluator attached. Gating would fall back to a lexical stub and H2 would "
        "be testing word counting rather than C2. Attach the G1 kernel output."
    )
print("EVALUATOR:", EVALUATOR)

from evaluator.score import T5Scorer

scorer = T5Scorer(EVALUATOR)
s = scorer.score(
    "The Kochi metro opened to the public in 2017.",
    ["The Kochi Metro began commercial operations in June 2017 after several delays.",
     "Penguins are flightless birds found mainly in the southern hemisphere."],
)
print(f"relevant {s[0]:+.3f}   irrelevant {s[1]:+.3f}")
assert s[0] > s[1], "evaluator is not separating - check the model"

## Endpoint for the verify, rewrite and decompose calls

In [ ]:
import shutil
import subprocess
import time
import urllib.request

if shutil.which("zstd") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                   shell=True, capture_output=True)
if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True, text=True)
OLLAMA = shutil.which("ollama")
assert OLLAMA, "ollama did not install"

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
except Exception:
    subprocess.Popen([OLLAMA, "serve"], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start")
print("ollama up", flush=True)
subprocess.run([OLLAMA, "pull", "qwen2.5:7b"], check=True)

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"
os.environ.setdefault("VERIMEM_API_KEY", "ollama")

COMMON = [
    "--data", "/kaggle/working/data/averitec/dev.json",
    "--store", "/kaggle/working/data/averitec/pool_dev.jsonl",
    "--trajectories", "/kaggle/working/data/trajectories/trajectories.jsonl",
    "--evaluator", str(EVALUATOR),
    "--misinfo", "/kaggle/working/data/averitec/misinfo_list.txt",
    "--targets", "100",
]

## Run 1 — the naive adversary, with each defence alone

All five configurations at all four rates. The nine cells from the previous sweep replay
from cache; the eight `gate_only` and `trust_only` cells are the new work.

Reading it: if `trust_only` alone flattens the curve, C3 is carrying H2 -- and since
these fakes cite unseen domains, that would be the prior doing the work rather than the
model. Run 2 is what tells those apart.

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "eval.run_poisoning", *COMMON,
     "--adversary", "naive", "--name", "poison-ablation-naive"],
    env=os.environ,
)
if r.returncode != 0:
    raise SystemExit("naive ablation sweep failed")

## Run 2 — the adaptive adversary

The fakes now cite the domains the clean history trusts most, so every poisoned record
clears the trust floor and C3 has nothing to object to. Only C2's evaluator gate is left,
and it has to decide on content: does this trajectory's evidence actually justify the
verdict it reached?

Two rates rather than four. The dose-response curve is established by run 1; what is
being asked here is whether the defence survives a stronger attacker at full strength, so
0% (the control) and 20% (the worst case) are the informative points and the two in
between would cost an hour to interpolate a line we already have the shape of.

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "eval.run_poisoning", *COMMON,
     "--adversary", "adaptive",
     "--rates", "0.0", "0.2",
     "--configs", "memory", "gate_only", "trust_only", "gated_memory",
     "--name", "poison-adaptive"],
    env=os.environ,
)
if r.returncode != 0:
    raise SystemExit("adaptive sweep failed")

## Both tables, printed to the log

In [ ]:
import json
import pathlib

for f in sorted(pathlib.Path(RESULTS).rglob("metrics.json")):
    if "poison" not in str(f):
        continue
    d = json.loads(f.read_text())
    print(f"\n=== {f.parent.name}: {d['n_targets']} targets, "
          f"{d['n_evaluated']} claims, adversary {d.get('adversary')} ===")
    if d.get("attack_domains"):
        print(f"    domains: {', '.join(d['attack_domains'][:6])}")
    print(f"  {'config':<14}{'rate':>6}{'ASR':>8}{'degraded':>10}"
          f"{'macro-F1':>10}{'poison kept':>13}")
    for c in d["cells"]:
        kept = c["retrieval"].get("poison_pass_rate")
        print(f"  {c['config']:<14}{c['rate']:>6}{c['attack_success_rate']:>8.3f}"
              f"{c['n_degraded']:>10}{c['macro_f1']:>10.3f}"
              f"{'-' if kept is None else format(kept, '.3f'):>13}")

In [ ]:
import shutil
shutil.rmtree("/kaggle/working/VeriMem/.git", ignore_errors=True)
!du -sh /kaggle/working/cache /kaggle/working/results